## Setup

### Install required packages

The code below implements this step in the prompt engineering workflow.

In [ ]:
# Install required packages
# !pip install openai anthropic langchain python-dotenv

**OpenAI API:** Interface to GPT and embedding models hosted by OpenAI. Requests are sent over HTTPS and billed per token.

- **Chat completions**: Send a conversation and get a model response
- **Embeddings**: Convert text into vector representations
- **Fine-tuning**: Customize model behavior with your own examples

Always handle API keys securely (environment variables, not hardcoded) and implement rate limiting and error handling.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def ask_llm(prompt, model="gpt-3.5-turbo", temperature=0.7):
    """Helper function to query LLM."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

## Example 1: Basic vs. Improved Prompt

### ❌ Vague prompt

✅ Specific prompt

In [ ]:
# ❌ Vague prompt
bad_prompt = "Tell me about Python"

# ✅ Specific prompt
good_prompt = """
Explain Python's list comprehensions to a beginner programmer.
Include:
1. What they are (1-2 sentences)
2. Basic syntax
3. One simple example
4. One common use case

Keep the explanation under 150 words.
"""

print("=== Vague Prompt ===")
print(ask_llm(bad_prompt)[:200], "...\n")

print("=== Specific Prompt ===")
print(ask_llm(good_prompt))

## Example 2: Few-Shot Learning

Run this cell and inspect the output to verify the prompt engineering operations produce the expected results.

In [ ]:
few_shot_prompt = """
Convert the following sentences to JSON format.

Example 1:
Input: "John lives in New York and works as a doctor."
Output: {"name": "John", "city": "New York", "occupation": "doctor"}

Example 2:
Input: "Sarah is from London and she is a teacher."
Output: {"name": "Sarah", "city": "London", "occupation": "teacher"}

Now convert this:
Input: "Mike lives in Tokyo and works as an engineer."
Output:
"""

print(ask_llm(few_shot_prompt, temperature=0))

## Example 3: Chain-of-Thought Reasoning

### Without CoT

With CoT

In [ ]:
# Without CoT
simple_prompt = """
A store had 20 apples. They sold 12 apples in the morning and 5 in the afternoon.
Then they received a delivery of 15 apples. How many apples do they have now?
"""

# With CoT
cot_prompt = simple_prompt + "\nLet's solve this step by step:"

print("=== Without Chain-of-Thought ===")
print(ask_llm(simple_prompt, temperature=0), "\n")

print("=== With Chain-of-Thought ===")
print(ask_llm(cot_prompt, temperature=0))

## Example 4: System Prompts

### Define `ask_with_system()`

Example: Expert vs. Beginner explanations

In [ ]:
def ask_with_system(system_prompt, user_prompt):
    """Use system prompt to set behavior."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

# Example: Expert vs. Beginner explanations
question = "What is recursion in programming?"

expert_system = "You are a computer science professor. Explain concepts with technical precision and examples."
beginner_system = "You are a patient teacher for absolute beginners. Use simple language and everyday analogies."

print("=== Expert Mode ===")
print(ask_with_system(expert_system, question), "\n")

print("=== Beginner Mode ===")
print(ask_with_system(beginner_system, question))

## Example 5: Structured Output

### Parse as JSON

Define variables and print their values to verify the results.

In [ ]:
structured_prompt = """
Analyze the following product review and extract information in this exact JSON format:

{
  "sentiment": "positive/negative/neutral",
  "rating_guess": "1-5",
  "key_points": ["point1", "point2"],
  "mentioned_features": ["feature1", "feature2"]
}

Review: "This laptop is amazing! Super fast processor and the battery lasts all day. 
Only complaint is it's a bit heavy to carry around."

Return ONLY the JSON, no other text:
"""

result = ask_llm(structured_prompt, temperature=0)
print(result)

# Parse as JSON
import json
data = json.loads(result)
print("\nParsed data:", data)

## Key Takeaways

### 1. **Be Specific**
- Clear instructions
- Defined format
- Constraints and requirements

### 2. **Use Examples (Few-Shot)**
- Show desired input/output
- 2-5 examples work well
- Consistent format

### 3. **Add Reasoning (CoT)**
- "Let's think step by step"
- "First..., then..., finally..."
- Better for complex tasks

### 4. **Set Context (System Prompts)**
- Define role and behavior
- Set tone and style
- Add constraints

### 5. **Structure Output**
- Specify exact format
- Use delimiters
- Request JSON/XML/etc.

## Next Steps

1. **`01_basic_prompting.ipynb`** - Master fundamental techniques
2. **`02_chain_of_thought.ipynb`** - Advanced reasoning
3. **`03_react_prompting.ipynb`** - Tool use and actions
4. **Practice!** - Try improving prompts you use daily

Remember: Prompt engineering is iterative. Test, analyze, and refine! 🚀